# Notebook 4: Advanced Patterns — State, Guardrails, and Discussion Prep

**Goal**: Go beyond the basic loop. These patterns demonstrate depth and will help in the post-coding discussion.

By the end, you'll know:
- How to manage state across tool calls
- How to add guardrails and validation
- Error handling strategies (is_error, retries)
- How to use tool_choice for control
- Trade-offs you'll discuss with the interviewer

---

In [ ]:
!pip install anthropic pydantic -q
import anthropic
import json
import re
from pydantic import BaseModel, Field
from typing import Literal

client = anthropic.Anthropic()

# Pydantic → Anthropic tool helper (from Notebook 1, section 1.4b)
def tool(model: type[BaseModel]):
    """Convert Pydantic model → Anthropic tool definition."""
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', model.__name__).lower()
    return {
        "name": name,
        "description": model.__doc__ or "",
        "input_schema": model.model_json_schema()
    }

print("Client + Pydantic tool() helper ready.")

## 4.1 Stateful Tools — Tools That Share Data

In real agents, tools often need to share state. For example:
- A shopping cart that tools add to and read from
- A database connection that persists across calls
- A scratchpad for intermediate computations

### Pattern: Use a class or closure to hold state

In [ ]:
class ShoppingAgent:
    """Agent with stateful tools — shopping cart example."""
    
    def __init__(self):
        self.cart = []  # Shared state
        self.catalog = {
            "LAPTOP-1": {"name": "MacBook Pro 14\"", "price": 1999.00, "stock": 5},
            "PHONE-1": {"name": "iPhone 16", "price": 999.00, "stock": 12},
            "TABLET-1": {"name": "iPad Air", "price": 599.00, "stock": 0},
            "HEADPH-1": {"name": "AirPods Pro", "price": 249.00, "stock": 30},
        }
        
        # Tool definitions
        self.tools = [
            {
                "name": "search_products",
                "description": "Search the product catalog by keyword. Returns matching products with IDs, names, prices, and stock status.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search keyword"}
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "add_to_cart",
                "description": "Add a product to the shopping cart by product ID. Checks stock availability. Returns the updated cart.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "product_id": {"type": "string", "description": "Product ID, e.g. LAPTOP-1"},
                        "quantity": {"type": "integer", "description": "How many to add. Default 1."}
                    },
                    "required": ["product_id"]
                }
            },
            {
                "name": "view_cart",
                "description": "View the current shopping cart contents and total price.",
                "input_schema": {"type": "object", "properties": {}}
            },
            {
                "name": "checkout",
                "description": "Complete the purchase. Only call when the user confirms they want to buy.",
                "input_schema": {"type": "object", "properties": {}}
            }
        ]
    
    # Tool implementations — they access self.cart and self.catalog
    def search_products(self, query):
        results = []
        for pid, product in self.catalog.items():
            if query.lower() in product["name"].lower():
                results.append({"id": pid, **product, "in_stock": product["stock"] > 0})
        return json.dumps({"results": results, "count": len(results)})
    
    def add_to_cart(self, product_id, quantity=1):
        product = self.catalog.get(product_id)
        if not product:
            return json.dumps({"error": f"Product {product_id} not found"})
        if product["stock"] < quantity:
            return json.dumps({"error": f"Not enough stock. Available: {product['stock']}"})
        
        self.cart.append({"product_id": product_id, "name": product["name"], 
                         "price": product["price"], "quantity": quantity})
        return json.dumps({"added": product["name"], "quantity": quantity, "cart_size": len(self.cart)})
    
    def view_cart(self):
        total = sum(item["price"] * item["quantity"] for item in self.cart)
        return json.dumps({"items": self.cart, "total": total, "item_count": len(self.cart)})
    
    def checkout(self):
        if not self.cart:
            return json.dumps({"error": "Cart is empty"})
        total = sum(item["price"] * item["quantity"] for item in self.cart)
        order_id = "ORD-" + str(hash(json.dumps(self.cart)))[-6:]
        result = {"order_id": order_id, "total": total, "items": len(self.cart), "status": "confirmed"}
        self.cart = []  # Clear cart after purchase
        return json.dumps(result)
    
    def process_tool_call(self, name, input_data):
        """Dispatch tool calls to methods."""
        handlers = {
            "search_products": self.search_products,
            "add_to_cart": self.add_to_cart,
            "view_cart": self.view_cart,
            "checkout": self.checkout,
        }
        handler = handlers.get(name)
        if not handler:
            return f"Unknown tool: {name}"
        try:
            return handler(**input_data)
        except Exception as e:
            return f"Error: {e}"
    
    def run(self, user_message, max_turns=10):
        """Run the agentic loop."""
        messages = [{"role": "user", "content": user_message}]
        
        for _ in range(max_turns):
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                tools=self.tools,
                system="You are a helpful shopping assistant. Help the user find and buy products.",
                messages=messages
            )
            
            if response.stop_reason == "end_turn":
                return next((b.text for b in response.content if b.type == "text"), "")
            
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = self.process_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            messages.append({"role": "user", "content": tool_results})
        
        return "Max turns reached"

print("ShoppingAgent defined.")

In [ ]:
# Test the stateful agent
agent = ShoppingAgent()

result = agent.run("I'm looking for some Apple headphones. Can you add them to my cart and show me the total?")
print(result)
print(f"\nCart state: {agent.cart}")

### Discussion point: Why use a class?

- **Encapsulation**: Cart state, catalog, and tools all live together
- **Clean dispatcher**: `self.search_products` etc. are methods
- **Testability**: You can create fresh instances for each test
- **Alternative**: You could use module-level globals or closures, but classes are cleaner for agents with shared state

## 4.2 Error Handling Strategies

Three levels of error handling you should know about:

In [ ]:
# ========== Level 1: Basic — return error string ==========
def process_tool_basic(name, input_data):
    try:
        return handlers[name](**input_data)
    except Exception as e:
        return f"Error: {e}"  # Claude sees this as a normal result


# ========== Level 2: Proper — use is_error flag ==========
def process_tool_proper(name, input_data):
    """Returns (result_string, is_error_bool)."""
    try:
        result = handlers[name](**input_data)
        return (str(result), False)
    except KeyError:
        return (f"Unknown tool: {name}", True)
    except TypeError as e:
        return (f"Invalid parameters: {e}", True)
    except Exception as e:
        return (f"Tool execution failed: {e}", True)


# Use in the loop:
# result, is_error = process_tool_proper(block.name, block.input)
# tool_results.append({
#     "type": "tool_result",
#     "tool_use_id": block.id,
#     "content": result,
#     **(({"is_error": True} if is_error else {}))
# })


# ========== Level 3: Advanced — with retry logic ==========
def run_agent_with_retries(user_message, tools, process_fn, max_turns=10, max_retries=2):
    """Agent loop that tracks and limits retries per tool."""
    messages = [{"role": "user", "content": user_message}]
    retry_counts = {}  # Track retries per tool_use_id
    
    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if b.type == "text"), "")
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                # Track retries
                key = block.name
                retry_counts[key] = retry_counts.get(key, 0) + 1
                
                if retry_counts[key] > max_retries:
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": f"Tool {block.name} has been called too many times. Please provide a response without using this tool.",
                        "is_error": True
                    })
                else:
                    result = process_fn(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
        
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"

print("Error handling patterns defined.")

## 4.3 Using tool_choice for Structured Extraction

A powerful pattern: use `tool_choice` to **force** Claude to return structured data in a specific format. This is like using tools as output schemas.

In [ ]:
# Pattern: Force structured output using a tool

extract_tool = {
    "name": "extract_entities",
    "description": "Extract named entities from text. Call this with the entities you find.",
    "input_schema": {
        "type": "object",
        "properties": {
            "people": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of person names found in the text"
            },
            "organizations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of organization names found in the text"
            },
            "locations": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of locations found in the text"
            }
        },
        "required": ["people", "organizations", "locations"]
    }
}

# Force Claude to use this specific tool
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[extract_tool],
    tool_choice={"type": "tool", "name": "extract_entities"},  # <-- FORCED
    messages=[{
        "role": "user",
        "content": "Tim Cook announced that Apple will open a new office in Austin, Texas, in partnership with Google and Microsoft."
    }]
)

# Extract the structured data
tool_block = next(b for b in response.content if b.type == "tool_use")
entities = tool_block.input

print("Extracted entities:")
print(json.dumps(entities, indent=2))

### Discussion point: Tools as output schemas

This is a useful trick for when you need structured data from Claude:
- Define a "tool" that is really just an output schema
- Force Claude to "call" it with `tool_choice: {"type": "tool", "name": "..."}`
- The `input` of the tool call IS your structured output
- You never actually execute the tool — you just use the data

In [ ]:
# ========== Pydantic makes structured extraction even cleaner ==========
# Compare: the extract_entities tool above was ~20 lines of JSON schema.
# With Pydantic, it's 5 lines:

class ExtractEntities(BaseModel):
    """Extract named entities from text. Call this with the entities you find."""
    people: list[str] = Field(description="Person names found in the text")
    organizations: list[str] = Field(description="Organization names found in the text")
    locations: list[str] = Field(description="Locations found in the text")

# Use it with tool_choice — identical pattern, less boilerplate
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[tool(ExtractEntities)],
    tool_choice={"type": "tool", "name": "extract_entities"},
    messages=[{
        "role": "user",
        "content": "Satya Nadella said Microsoft and OpenAI will expand operations in London and Tokyo."
    }]
)

entities = next(b for b in response.content if b.type == "tool_use").input
print("Pydantic-powered extraction:")
print(json.dumps(entities, indent=2))

## 4.4 Multi-Turn Conversations with Tool Use

Real agents often need to maintain a conversation across multiple user inputs, not just one.

In [ ]:
class ConversationalAgent:
    """Agent that maintains conversation history across multiple user turns."""
    
    def __init__(self, tools, process_tool_call, system_prompt=None):
        self.tools = tools
        self.process_tool_call = process_tool_call
        self.system_prompt = system_prompt
        self.messages = []  # Persistent conversation history
    
    def chat(self, user_message, max_turns=10):
        """Send a message and get a response, maintaining conversation context."""
        self.messages.append({"role": "user", "content": user_message})
        
        for _ in range(max_turns):
            kwargs = {
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 4096,
                "tools": self.tools,
                "messages": self.messages,
            }
            if self.system_prompt:
                kwargs["system"] = self.system_prompt
            
            response = client.messages.create(**kwargs)
            
            if response.stop_reason == "end_turn":
                # Add Claude's final response to history
                self.messages.append({"role": "assistant", "content": response.content})
                return next((b.text for b in response.content if b.type == "text"), "")
            
            # Tool use — process and continue
            self.messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = self.process_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            self.messages.append({"role": "user", "content": tool_results})
        
        return "Max turns reached"

print("ConversationalAgent defined.")

In [ ]:
# Test multi-turn conversation
def fake_weather(location, unit="fahrenheit"):
    return json.dumps({"location": location, "temp": 62, "condition": "foggy"})

def fake_calculator(operation, a, b):
    ops = {"add": a+b, "subtract": a-b, "multiply": a*b, "divide": a/b if b else 0}
    return str(ops.get(operation, "?"))

conv_tools = [
    {"name": "get_weather", "description": "Get weather for a location.",
     "input_schema": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}},
    {"name": "calculator", "description": "Do math: add, subtract, multiply, divide.",
     "input_schema": {"type": "object", "properties": {"operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]}, "a": {"type": "number"}, "b": {"type": "number"}}, "required": ["operation", "a", "b"]}}
]

def conv_dispatch(name, inp):
    return {"get_weather": fake_weather, "calculator": fake_calculator}.get(name, lambda **x: "unknown")(**inp)

agent = ConversationalAgent(conv_tools, conv_dispatch)

# Turn 1
print("User: What's the weather in SF?")
print(f"Claude: {agent.chat('What is the weather in San Francisco?')}")

# Turn 2 — Claude remembers the previous context
print(f"\nUser: Convert that to celsius")
print(f"Claude: {agent.chat('Can you convert that temperature to celsius?')}")

# Turn 3
print(f"\nUser: What is 42 * 17?")
print(f"Claude: {agent.chat('And what is 42 times 17?')}")

## 4.4b Interactive Chatbot — `input()` Loop in Notebook

This is the pattern for making an agent you can actually **chat with** inside Colab/Jupyter. It combines the `ConversationalAgent` with Python's `input()` to create a REPL-style chatbot.

Key insight: `input()` works in both Colab and Jupyter — it renders a text field in the notebook output.

In [ ]:
# ============================================
# INTERACTIVE CHATBOT — Run this cell and chat!
# ============================================
# Uses the ConversationalAgent from 4.4 above.
# Type your messages, press Enter. Type "quit" or "exit" to stop.
# Works in both Colab and Jupyter notebooks.

def interactive_chatbot(tools, process_tool_call, system_prompt=None):
    """
    Interactive chatbot loop using input().
    
    This is the pattern for making an agent you can talk to 
    in a notebook. Combines:
      - ConversationalAgent (persistent message history)
      - Python input() for user input
      - Agentic tool use loop inside each turn
    """
    messages = []  # Persistent conversation history
    
    print("=" * 50)
    print("  INTERACTIVE AGENT CHATBOT")
    print("  Type your message and press Enter.")
    print("  Type 'quit' or 'exit' to stop.")
    print("  Type 'history' to see message count.")
    print("  Type 'reset' to clear conversation.")
    print("=" * 50)
    
    while True:
        # Get user input
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nGoodbye!")
            break
        
        if not user_input:
            continue
        
        # Special commands
        if user_input.lower() in ("quit", "exit"):
            print("\nGoodbye!")
            break
        
        if user_input.lower() == "history":
            print(f"  [{len(messages)} messages in history]")
            continue
        
        if user_input.lower() == "reset":
            messages = []
            print("  [Conversation reset]")
            continue
        
        # Add user message to history
        messages.append({"role": "user", "content": user_input})
        
        # Agentic loop — handle tool calls until Claude gives a final answer
        for turn in range(10):  # max 10 tool-call rounds per user message
            kwargs = {
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 4096,
                "tools": tools,
                "messages": messages,
            }
            if system_prompt:
                kwargs["system"] = system_prompt
            
            response = client.messages.create(**kwargs)
            
            # Final answer — print and save to history
            if response.stop_reason == "end_turn":
                messages.append({"role": "assistant", "content": response.content})
                answer = next((b.text for b in response.content if b.type == "text"), "")
                print(f"\nAssistant: {answer}")
                break
            
            # Tool use — execute silently and continue
            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"  [calling {block.name}...]")
                        result = process_tool_call(block.name, block.input)
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result)
                        })
                
                messages.append({"role": "user", "content": tool_results})
    
    return messages  # Return history for inspection


# ---- Launch the chatbot with the weather/calculator tools ----
# (Uses the tools and dispatcher defined in section 4.4 above)
#
# Uncomment and run to start chatting:

# chat_history = interactive_chatbot(
#     tools=conv_tools,
#     process_tool_call=conv_dispatch,
#     system_prompt="You are a helpful assistant with access to weather and calculator tools."
# )

In [ ]:
# ============================================
# INTERACTIVE SHOPPING CHATBOT — Stateful version
# ============================================
# This shows how to plug a CLASS-BASED agent into the input() loop.
# The ShoppingAgent (from 4.1) keeps cart state across turns.
#
# Try a conversation like:
#   "What products do you have?"
#   "Add the AirPods to my cart"
#   "What's in my cart?"
#   "Checkout"

def interactive_shopping_chatbot():
    """Interactive chatbot using the ShoppingAgent with persistent cart state."""
    shop = ShoppingAgent()
    messages = []
    
    print("=" * 50)
    print("  SHOPPING ASSISTANT (stateful)")
    print("  Products: MacBook Pro, iPhone 16, iPad Air, AirPods Pro")
    print("  Type 'quit' to exit, 'cart' to see cart state")
    print("=" * 50)
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit"):
            print("\nGoodbye!")
            break
        if user_input.lower() == "cart":
            print(f"  [Internal cart state: {shop.cart}]")
            continue
        
        # Add to persistent history
        messages.append({"role": "user", "content": user_input})
        
        # Agentic loop for this turn
        for _ in range(10):
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                tools=shop.tools,
                system="You are a shopping assistant. Help the user browse, add items to cart, and checkout.",
                messages=messages
            )
            
            if response.stop_reason == "end_turn":
                messages.append({"role": "assistant", "content": response.content})
                print(f"\nAssistant: {next((b.text for b in response.content if b.type == 'text'), '')}")
                break
            
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  [calling {block.name}...]")
                    result = shop.process_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})

# Uncomment to run:
# interactive_shopping_chatbot()

## 4.5 Design Trade-offs — Interview Discussion Prep

The interviewer will ask about trade-offs. Here are the key ones:

### 1. Tool Description Verbosity vs. Token Cost
- **More detail** → Claude uses tools more accurately → costs more tokens
- **Less detail** → Claude may misuse tools → fewer tokens per call
- **Best practice**: Err on the side of verbose descriptions. The accuracy gain outweighs the cost.

### 2. `max_turns` — How High?
- Too low (2-3) → Agent can't do multi-step tasks
- Too high (100) → Risk of runaway loops, high cost
- **Recommended**: 10-15 for most tasks. Adjust based on expected complexity.

### 3. Error Handling — Fail Fast vs. Let Claude Recover
- **Fail fast**: Raise exception on error → agent stops
- **Let Claude recover**: Return error as `is_error: true` → Claude can retry/adapt
- **Best practice**: Let Claude recover for user-facing agents. Fail fast for internal/batch agents.

### 4. State Management — Global vs. Scoped
- **Global state** (module variables): Simple, but not testable or thread-safe
- **Class/instance state**: Clean, testable, but more boilerplate
- **Closure state**: Functional approach, compact but less readable
- **Best practice**: Class for anything beyond toy examples.

### 5. Tool Design — Many Specific Tools vs. Few General Tools
- **Many specific** (e.g., `get_order_status`, `get_order_items`, `get_order_tracking`): Clear intent, Claude uses them precisely
- **Few general** (e.g., `query_database` with a query parameter): Flexible but Claude may construct wrong queries
- **Best practice**: Start with specific tools. Only generalize if you hit the tool limit.

### 6. Model Choice
- **Opus 4.6**: Best for complex tool use, ambiguous queries, multi-tool scenarios. Asks for clarification.
- **Sonnet 4**: Good balance. Use `claude-sonnet-4-20250514` for interview (fast + capable).
- **Haiku 4.5**: Fastest, cheapest. May infer missing params instead of asking.

## 4.6 Validation and Guardrails

In [ ]:
# Pattern: Input validation before executing tools

def validated_process_tool(name, input_data):
    """Process tool call with input validation."""
    
    # Validation rules per tool
    if name == "delete_user":
        # Dangerous operation — require confirmation
        if not input_data.get("confirmed"):
            return json.dumps({"error": "Deletion requires confirmed=true parameter"})
    
    if name == "send_email":
        # Validate email format
        email = input_data.get("to", "")
        if "@" not in email:
            return json.dumps({"error": f"Invalid email address: {email}"})
    
    if name == "run_query":
        # SQL injection protection
        query = input_data.get("query", "")
        dangerous = ["DROP", "DELETE", "TRUNCATE", "ALTER"]
        if any(d in query.upper() for d in dangerous):
            return json.dumps({"error": "Destructive queries are not allowed"})
    
    # If validation passes, execute normally
    return handlers.get(name, lambda **x: "Unknown tool")(**input_data)

print("Validated dispatcher defined.")

## 4.7 Pattern: Logging and Observability

In the interview discussion, mentioning observability shows maturity.

In [ ]:
def run_agent_with_logging(user_message, tools, process_fn, max_turns=10):
    """Agent loop with full logging for debugging."""
    messages = [{"role": "user", "content": user_message}]
    
    log = {
        "user_message": user_message,
        "turns": [],
        "total_input_tokens": 0,
        "total_output_tokens": 0,
    }
    
    for turn_num in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        # Track tokens
        log["total_input_tokens"] += response.usage.input_tokens
        log["total_output_tokens"] += response.usage.output_tokens
        
        turn_log = {
            "turn": turn_num + 1,
            "stop_reason": response.stop_reason,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "tool_calls": []
        }
        
        if response.stop_reason == "end_turn":
            final_text = next((b.text for b in response.content if b.type == "text"), "")
            turn_log["final_response"] = final_text[:100]
            log["turns"].append(turn_log)
            log["final_answer"] = final_text
            
            print("\n=== Agent Log ===")
            print(f"Turns: {len(log['turns'])}")
            print(f"Total tokens: {log['total_input_tokens']} in / {log['total_output_tokens']} out")
            for t in log["turns"]:
                print(f"  Turn {t['turn']}: {t['stop_reason']}, tools: {t['tool_calls']}")
            print("================\n")
            
            return final_text
        
        messages.append({"role": "assistant", "content": response.content})
        
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = process_fn(block.name, block.input)
                turn_log["tool_calls"].append({
                    "name": block.name,
                    "input": block.input,
                    "result": str(result)[:100]
                })
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        
        log["turns"].append(turn_log)
        messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached"

print("Logging agent defined.")

## 4.8 Exercise: Build a File System Agent

Build an agent with these tools (all fake implementations):
1. `list_directory` — takes `path`, returns list of files
2. `read_file` — takes `path`, returns file contents
3. `write_file` — takes `path` and `content`, writes file
4. `search_files` — takes `pattern`, returns matching file paths

Include:
- Stateful file system (dict)
- Validation (don't allow writing to system paths)
- The agentic loop

Test: "Find all Python files and show me what's in main.py"

In [ ]:
# YOUR ANSWER HERE


In [ ]:
# SOLUTION

class FileSystemAgent:
    def __init__(self):
        # Fake file system
        self.fs = {
            "/home/user/project/main.py": "def main():\n    print('Hello world')\n\nif __name__ == '__main__':\n    main()",
            "/home/user/project/utils.py": "def add(a, b):\n    return a + b\n",
            "/home/user/project/README.md": "# My Project\n\nA sample project.",
            "/home/user/project/data/config.json": '{"debug": true, "port": 8080}',
            "/home/user/project/tests/test_main.py": "def test_main():\n    assert True\n",
        }
        
        self.tools = [
            {
                "name": "list_directory",
                "description": "List files and subdirectories in a directory path. Returns names of all items.",
                "input_schema": {
                    "type": "object",
                    "properties": {"path": {"type": "string", "description": "Directory path to list"}},
                    "required": ["path"]
                }
            },
            {
                "name": "read_file",
                "description": "Read the contents of a file. Returns the file text.",
                "input_schema": {
                    "type": "object",
                    "properties": {"path": {"type": "string", "description": "Full file path"}},
                    "required": ["path"]
                }
            },
            {
                "name": "write_file",
                "description": "Write content to a file. Creates or overwrites.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "File path to write to"},
                        "content": {"type": "string", "description": "Content to write"}
                    },
                    "required": ["path", "content"]
                }
            },
            {
                "name": "search_files",
                "description": "Search for files matching a pattern (e.g. *.py). Returns matching file paths.",
                "input_schema": {
                    "type": "object",
                    "properties": {"pattern": {"type": "string", "description": "Search pattern, e.g. *.py"}},
                    "required": ["pattern"]
                }
            }
        ]
    
    def list_directory(self, path):
        if not path.endswith("/"): path += "/"
        items = set()
        for fpath in self.fs:
            if fpath.startswith(path):
                remaining = fpath[len(path):]
                items.add(remaining.split("/")[0])
        return json.dumps({"path": path, "items": sorted(items)})
    
    def read_file(self, path):
        if path in self.fs:
            return json.dumps({"path": path, "content": self.fs[path]})
        return json.dumps({"error": f"File not found: {path}"})
    
    def write_file(self, path, content):
        # Validation: don't allow system paths
        if path.startswith("/etc/") or path.startswith("/sys/"):
            return json.dumps({"error": "Cannot write to system directories"})
        self.fs[path] = content
        return json.dumps({"status": "written", "path": path, "bytes": len(content)})
    
    def search_files(self, pattern):
        import fnmatch
        matches = [p for p in self.fs if fnmatch.fnmatch(p.split("/")[-1], pattern)]
        return json.dumps({"pattern": pattern, "matches": matches})
    
    def process_tool(self, name, inp):
        h = {"list_directory": self.list_directory, "read_file": self.read_file,
             "write_file": self.write_file, "search_files": self.search_files}
        try:
            return h[name](**inp)
        except Exception as e:
            return f"Error: {e}"
    
    def run(self, user_message, max_turns=10):
        messages = [{"role": "user", "content": user_message}]
        for _ in range(max_turns):
            response = client.messages.create(
                model="claude-sonnet-4-20250514", max_tokens=4096,
                tools=self.tools, messages=messages
            )
            if response.stop_reason == "end_turn":
                return next((b.text for b in response.content if b.type == "text"), "")
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = self.process_tool(block.name, block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
            messages.append({"role": "user", "content": tool_results})
        return "Max turns reached"

# Test
fs_agent = FileSystemAgent()
print(fs_agent.run("Find all Python files and show me what's in main.py"))

## 4.9 Streaming with Tool Use

Streaming lets Claude's text appear **token-by-token** as it's generated — much better UX for chatbots. The API fully supports streaming with tool use, and it might come up as a curveball question.

### Two Ways to Stream

| Method | How | Returns |
|--------|-----|---------|
| `client.messages.stream()` | Context manager (recommended) | `MessageStream` helper object |
| `client.messages.create(stream=True)` | Raw parameter | Raw `MessageStreamEvent` iterator |

### The Escape Hatch: `get_final_message()`

Even when streaming, you can call `stream.get_final_message()` at the end to get the **exact same `Message` object** as a non-streaming call. This means your agentic loop logic (checking `stop_reason`, iterating `content` blocks) stays identical — you just add streaming on top.

In [ ]:
# ========== BASIC STREAMING — stream text output ==========
# The simplest pattern: stream text, then get the final message object.

# Stream a simple text response (no tool use triggered)
with client.messages.stream(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly 10 words."}]
) as stream:
    print("Streaming text: ", end="")
    for text in stream.text_stream:
        print(text, end="", flush=True)  # Token-by-token output
    print()

# get_final_message() → IDENTICAL to client.messages.create() result
final = stream.get_final_message()
print(f"\nstop_reason: {final.stop_reason}")
print(f"Usage: {final.usage.input_tokens} in / {final.usage.output_tokens} out")
print(f"Content blocks: {[b.type for b in final.content]}")

In [ ]:
# ========== STREAMING AGENTIC LOOP — drop-in replacement ==========
#
# KEY INSIGHT: The ONLY change from the standard loop is:
#   client.messages.create()  →  client.messages.stream()
# Everything else (stop_reason, tool processing) stays IDENTICAL
# thanks to get_final_message().

def run_agent_streaming(user_message, tools, process_tool_call, max_turns=10):
    """Streaming agentic loop — streams text, handles tool calls identically."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        # ---- THIS IS THE ONLY DIFFERENCE ----
        with client.messages.stream(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages
        ) as stream:
            # Stream text tokens as they arrive
            for text in stream.text_stream:
                print(text, end="", flush=True)

            # get_final_message() → same Message object as .create()
            response = stream.get_final_message()
        # ---- END OF DIFFERENCE ----

        # From here: IDENTICAL to non-streaming loop
        if response.stop_reason == "end_turn":
            print()  # Newline after streamed text
            return next((b.text for b in response.content if b.type == "text"), "")

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\n  [calling {block.name}...]")
                result = process_tool_call(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })

        messages.append({"role": "user", "content": tool_results})

    return "Max turns reached"

print("run_agent_streaming() defined — drop-in replacement for run_agent()")

In [ ]:
# ========== TEST: Streaming agentic loop with tool use ==========
# Uses the weather/calculator tools from section 4.4.
# Watch the text stream in real-time, then see tool calls happen.

result = run_agent_streaming(
    user_message="What's the weather in Tokyo? Also what's 99 * 77?",
    tools=conv_tools,
    process_tool_call=conv_dispatch
)
print(f"\n--- Final result length: {len(result)} chars ---")

In [ ]:
# ========== RAW EVENT ITERATION (advanced / discussion only) ==========
# If the interviewer asks about low-level streaming events, here's the pattern.
# You probably won't need this — get_final_message() covers 99% of cases.
# But knowing the event types shows depth.

# Event types in order:
#   message_start        → contains message metadata (id, model, role)
#   content_block_start  → new content block beginning (text or tool_use)
#   content_block_delta  → incremental data (text_delta or input_json_delta)
#   content_block_stop   → block finished
#   message_delta        → final metadata (stop_reason, usage)
#   message_stop         → stream complete

# Example: iterate raw events
with client.messages.stream(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hi briefly."}]
) as stream:
    for event in stream:
        # event.type is one of the types above
        print(f"  {event.type}", end="")
        if hasattr(event, 'delta') and hasattr(event.delta, 'text'):
            print(f" → '{event.delta.text}'", end="")
        if hasattr(event, 'delta') and hasattr(event.delta, 'stop_reason'):
            print(f" → stop_reason={event.delta.stop_reason}", end="")
        print()

print("\n--- Key stream helpers ---")
print("stream.text_stream        → yields text deltas only (for printing)")
print("stream.get_final_message() → returns complete Message object")
print("stream.get_final_text()    → returns just the text content")

---

## Summary — Advanced Patterns

1. **Stateful tools** → Use a class with instance state
2. **Error handling** → Return `is_error: true`, let Claude recover
3. **tool_choice** → Force specific tools for structured output
4. **Multi-turn** → Persist `messages` list across user turns
5. **Validation** → Check inputs before executing dangerous operations
6. **Logging** → Track turns, tokens, tool calls for debugging
7. **Streaming** → `client.messages.stream()` + `get_final_message()` = drop-in replacement

**Next: Notebook 5 — Mock Interview (timed practice)**